# Save dataset to training, validation and test sets

In [1]:
# imports
from pathlib import Path
import pandas as pd
import json
import sys

# helpers
sys.path.insert(0, "../../")
from geometric_extraction_helper import (
    ALL_FEATURE_KEYS
)

from dataloader import (
    create_hf_dataset,
    show_label_distribution,
    oversample_training_data,
)

from models_helper import ModelTrainer
from dataviz_helper import plot_learning_curves

In [2]:
# data directory and output path
DATA_DIR = "../3_5_clean_all_features/data_dropped_input_duplicates.parquet"
OUTPUT_PATH = "./data_original.parquet"
DATASET_NAME = "dataset"

# load parquet file
df = pd.read_parquet(DATA_DIR)
df.head()

,ifc_file_path,model_name,ifc_schema,guid,label_ifc_entity,label_predefined_type,storey,type_gks,eBKPh,excel_label_file_path,...,mat_kunststein,mat_kunststoff,mat_metall,mat_mörtel,mat_naturstein,mat_putz,mat_stahl,mat_zement,horizontal_elements_above,horizontal_elements_below
0,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_VOL_...,ZUST_P_ARC_VOL_,IFC4,1HyWKnRVf2nQPdSu_qBpKS,IfcSpace,GFA,U01,GF,"[C Konstruktion Gebäude, C05 Ergänzende Leistu...",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_VOL_...,ZUST_P_ARC_VOL_,IFC4,330umOG8j8lh$KH3zRhUgt,IfcSpace,GFA,U01,GF,"[C Konstruktion Gebäude, C05 Ergänzende Leistu...",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_VOL_...,ZUST_P_ARC_VOL_,IFC4,0RuGd4rvD2xeZxAp7YWBVl,IfcSpace,GFA,U01,GF,"[C Konstruktion Gebäude, C05 Ergänzende Leistu...",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_EBK_...,ZUST_P_ARC_EBK_,IFC4,0p4jVtekv8bero8khCpNIT,IfcColumn,NOTDEFINED,U01,"Beton, Stahlbeton Priorität 800 D300","[C03 Stützenkonstruktion, C03.02 Innenstütze]",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,-1.0
4,../../0_data/1_ifc_models/ZUST/ZUST_P_ARC_EBK_...,ZUST_P_ARC_EBK_,IFC4,2YC9riJTTCN89fdfN7wHz9,IfcColumn,NOTDEFINED,U01,"Beton, Stahlbeton D300","[C03 Stützenkonstruktion, C03.02 Innenstütze]",../../0_data/1_ifc_models/ZUST/ZUST_Alle_ARC_A...,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,-1.0


## 1. Fill NaN values with the category "unknown" to label and minor label columns

In [3]:
# show main labels
label_cols = [c for c in df.columns if c.startswith("label_")]
label_cols

['label_ifc_entity',
 'label_predefined_type',
 'label_is_external',
 'label_load_bearing']

In [4]:
# fill NaN values with the category "unknown" (for some elements it is not important for the eBKP-H classification)
df[label_cols] = df[label_cols].fillna("unknown")

In [5]:
# show minor labels
minor_label_cols = [c for c in df.columns if c.startswith("minor_")]
minor_label_cols

['minor_label_unter_terrain',
 'minor_label_konstruktionsergaenzung',
 'minor_label_deckbelag',
 'minor_label_bekleidung',
 'minor_label_aussenliegendes_bauteil',
 'minor_label_erdverbunden',
 'minor_label_unterkonstruktion',
 'minor_label_verdunkelung',
 'minor_label_schutzschicht',
 'minor_label_sonnenschutz',
 'minor_label_einbau',
 'minor_label_aufzugstyp']

In [6]:
# fill NaN values with the category "irrelevant" (for some elements it is not important for the eBKP-H classification)
minor_label_cols
df[minor_label_cols] = df[minor_label_cols].fillna("unknown")

## 2. Remove classes with less than 200 samples

In [7]:
# set the threshold of minimum samples per class to 200, it was researched in the threshold analysis notebook
min_samples = 200

# remove classes with less than min_samples
for col in label_cols + minor_label_cols:
    class_counts = df[col].value_counts()
    valid_classes = class_counts[class_counts >= min_samples].index
    removed = (~df[col].isin(valid_classes)).sum()
    if removed > 0:
        df = df[df[col].isin(valid_classes)].reset_index(drop=True)
        print(f"{col}: {removed} rows removed with following classes: {set(class_counts.index) - set(valid_classes)}")

label_ifc_entity: 111 rows removed with following classes: {'IfcTransportElement', 'IfcRamp', 'IfcWasteTerminal', 'IfcFooting'}
label_predefined_type: 911 rows removed with following classes: {'EXTERNAL', 'SHOWER', 'STRAIGHT', 'PARKING', 'ROOF', 'SINK', 'LIGHTDOME', 'ELEMENTEDWALL', 'BATH', 'CEILING', 'MOVABLE', 'SKYLIGHT', 'LANDING', 'GATE'}
minor_label_konstruktionsergaenzung: 473 rows removed with following classes: {'strassenfassade', 'hinterlüftete fassade (gedämmt+largo)', 'fassade bestehend sockel eg', 'hoffassade', 'fassade bestehend', 'hinterlüftete fassade (ungedämmt)', 'fassade hofgebäude', 'pv brüstungsergänzung', 'verputzt', 'fassade waschbeton', 'unbehandelt', 'dämmung fassade verputzt', 'hinterlüftete fassade (gedämmt)', 'flachdach best. gefällsschicht', 'geneigtes dach'}
minor_label_unterkonstruktion: 75 rows removed with following classes: {'true'}
minor_label_verdunkelung: 21 rows removed with following classes: {'true'}
minor_label_schutzschicht: 71 rows removed with

In [8]:
# drop columns that have become single-class after removing rare samples
for col in label_cols + minor_label_cols:
    if col in df.columns and df[col].nunique() == 1:
        df.drop(columns=[col], inplace=True)
        print(f'"{col}" label removed, because it consists less than < {min_samples} Samples.')

"minor_label_erdverbunden" label removed, because it consists less than < 200 Samples.
"minor_label_unterkonstruktion" label removed, because it consists less than < 200 Samples.
"minor_label_verdunkelung" label removed, because it consists less than < 200 Samples.
"minor_label_schutzschicht" label removed, because it consists less than < 200 Samples.
"minor_label_einbau" label removed, because it consists less than < 200 Samples.
"minor_label_aufzugstyp" label removed, because it consists less than < 200 Samples.


In [9]:
# update label lists after dropping single-class columns
label_cols = [c for c in label_cols if c in df.columns]
minor_label_cols = [c for c in minor_label_cols if c in df.columns]

In [10]:
print(f"\nRemaining data size: {len(df)}")


Remaining data size: 41463


## 3. Prepare and split the data

In [11]:
# create dataset with all main and minor labels, stratified on both with removing rare classes
ds, project_info = create_hf_dataset(
    df, ALL_FEATURE_KEYS,
    target_cols=label_cols + minor_label_cols,
    minor_cols=minor_label_cols,
    label_cols=label_cols,
)

Count of projects in splits: 8 train | 3 val | 2 test
Projects in train split: ['CHLI' 'ESAK' 'GERB' 'IMBU' 'KEHO' 'RALU' 'ZEGA' 'ZUST']
Projects in val split: ['BRUT' 'GSRH' 'KEPR']
Projects in test split: ['ADEM' 'LUMU']

Count of elements in splits: 27,695 | 10,219 | 3,549
Leakage deduplication: 0 rows from validation ds removed, 0 rows from test ds removed.
Orphan removal: 718 train | 1761 val | 0 test rows removed.

Train: 26977 (65.1%) | Val: 8458 (20.4%) | Test: 3549 (8.6%)


## 4. Analyse the distribution of the classes in the splitted datasets

In [12]:
# show distribution of main labels after removing rare classes
show_label_distribution(ds, label_cols)

train  train_pct  validation  \
feature               value                                               
label_ifc_entity      IfcColumn              454        1.7          73   
                      IfcCovering            330        1.2          37   
                      IfcDoor                731        2.7         392   
                      IfcFurniture             2        0.0          68   
                      IfcPlate               404        1.5         732   
                      IfcRailing            6938       25.7         169   
                      IfcRoof                456        1.7         308   
                      IfcSanitaryTerminal    329        1.2         555   
                      IfcSlab                920        3.4         805   
                      IfcSpace              2830       10.5         662   
                      IfcStairFlight          95        0.4          35   
                      IfcWall              12314       45.6        4338   
                      IfcWindow             1174        4.4         284   
label_predefined_type BASESLAB               261        1.0          31   
                      COLUMN                 300        1.1          63   
                      DOOR                   731        2.7         392   
                      FLAT_ROOF              456        1.7         308   
                      FLOOR                  659        2.4         774   
                      FLOORING               330        1.2          37   
                      GFA                    126        0.5         136   
                      GUARDRAIL             6938       25.7         169   
                      INTERNAL              2704       10.0         526   
                      NOTDEFINED            3637       13.5         276   
                      PARTITIONING           480        1.8         651   
                      PLUMBINGWALL          1399        5.2         501   
                      SHEET                  404        1.5         732   
                      SOLIDWALL             7359       27.3        3023   
                      TOILETPAN                8        0.0         236   
                      WASHHANDBASIN           11        0.0         319   
                      WINDOW                1174        4.4         284   
label_is_external     false                 9391       34.8        3861   
                      true                 14329       53.1        3277   
                      unknown               3257       12.1        1320   
label_load_bearing    false                 4251       15.8        3070   
                      true                 10276       38.1        3183   
                      unknown              12450       46.2        2205   

                                           validation_pct  test  test_pct  
feature               value                                                
label_ifc_entity      IfcColumn                       0.9    46       1.3  
                      IfcCovering                     0.4    69       1.9  
                      IfcDoor                         4.6   137       3.9  
                      IfcFurniture                    0.8     2       0.1  
                      IfcPlate                        8.7    26       0.7  
                      IfcRailing                      2.0    44       1.2  
                      IfcRoof                         3.6    67       1.9  
                      IfcSanitaryTerminal             6.6    76       2.1  
                      IfcSlab                         9.5   219       6.2  
                      IfcSpace                        7.8   684      19.3  
                      IfcStairFlight                  0.4   113       3.2  
                      IfcWall                        51.3  1807      50.9  
                      IfcWindow                       3.4   259       7.3  
label_predefined_type BASESLAB                      

## 5. Save the datasets

In [13]:
# save dataset as parquet files for with removed rare classes
for split in ["train", "validation", "test"]:
    ds[split].to_parquet(f"./{DATASET_NAME}_{split}_rare_classes_removed.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

## 6. Analyze optimal threshold for oversampling the training set

In [14]:
# use cleaned dataset for oversampling experiment
df_train_data = ds["train"].to_pandas()
df_val_data   = ds["validation"].to_pandas()

# get main label columns for oversampling
label_cols_for_oversampling = [c for c in df_train_data.columns if c.startswith("label_")]

# prepare validation data
X_val = df_val_data[ALL_FEATURE_KEYS]
y_val = df_val_data[label_cols_for_oversampling]

# candidate target counts to evaluate
target_counts = [0, 25, 50, 75, 100, 125, 150, 200, 300, 500, 750, 1000]

# collect f1_macro per label and per target_count
label_sizes  = {col: [] for col in label_cols_for_oversampling}
label_scores = {col: [] for col in label_cols_for_oversampling}
mean_scores  = []

# iterate over candidate target counts
for tc in target_counts:
    df_over = oversample_training_data(df_train_data, label_cols=label_cols_for_oversampling, target_count=tc)

    # train basic rf model on oversampled data
    trainer = ModelTrainer("random_forest", label_cols=label_cols_for_oversampling)
    trainer.fit(df_over[ALL_FEATURE_KEYS], df_over[label_cols_for_oversampling])

    # evaluate on validation set
    eval_df = trainer.evaluate(X_val, y_val)
    print(f"target_count = {tc}, mean F1-macro = {eval_df.loc['mean', 'f1_macro']:.4f}")

    # collect results
    for col in label_cols_for_oversampling:
        label_sizes[col].append(tc)
        label_scores[col].append(eval_df.loc[col, "f1_macro"])
    mean_scores.append(eval_df.loc["mean", "f1_macro"])

target_count = 0, mean F1-macro = 0.6957
target_count = 25, mean F1-macro = 0.6980
target_count = 50, mean F1-macro = 0.7041
target_count = 75, mean F1-macro = 0.6930
target_count = 100, mean F1-macro = 0.7062
target_count = 125, mean F1-macro = 0.6947
target_count = 150, mean F1-macro = 0.7052
target_count = 200, mean F1-macro = 0.6966
target_count = 300, mean F1-macro = 0.6758
target_count = 500, mean F1-macro = 0.6951
target_count = 750, mean F1-macro = 0.7034
target_count = 1000, mean F1-macro = 0.6888


In [15]:
# plot results
summary = {col.replace("label_", ""): label_scores[col] for col in label_cols_for_oversampling}
summary["mean"] = mean_scores
pd.DataFrame(summary, index=target_counts).rename_axis("target_count")

,ifc_entity,predefined_type,is_external,load_bearing,mean
target_count,,,,,
0,0.6274,0.4750,0.8537,0.8269,0.6957
25,0.6354,0.4837,0.8499,0.8230,0.6980
50,0.6438,0.5019,0.8439,0.8266,0.7041
75,0.6287,0.4803,0.8412,0.8217,0.6930
100,0.6446,0.4966,0.8620,0.8215,0.7062
125,0.6179,0.4916,0.8435,0.8259,0.6947
150,0.6357,0.4952,0.8632,0.8266,0.7052
200,0.6171,0.4771,0.8699,0.8221,0.6966
300,0.5599,0.4810,0.8407,0.8217,0.6758


In [16]:
# visualize results to decide on a good oversampling threshold for the oversampling trainingset
results_plot = {col.replace("label_", ""): (label_sizes[col], label_scores[col]) for col in label_cols_for_oversampling}
results_plot["mean"] = (target_counts, mean_scores)

plot_learning_curves(
    results_plot,
    label_name="Oversampling Schwellwert",
    x_max=1000,
    legend_rows=2,
    save="oversampling_threshold_analysis",
    chapter="implementation",
    threshold=100,
    yaxis=[0.4, 1.0],
)

figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/4 implementation/img/oversampling_threshold_analysis.svg


## 7. Oversampling the training set as experiment

In [17]:
# oversampling the training set as experiment
# based on the threshold in the previous section the ideal number of samples per class is 100
df_train_original = ds["train"].to_pandas()

label_cols_for_oversampling = [c for c in df_train_original.columns if c.startswith("label_")]
df_train_oversampled = oversample_training_data(
    df_train_original,
    label_cols=label_cols_for_oversampling,
    target_count=100,
)

In [18]:
# save oversampled training set (val/test remain the original splits)
df_train_oversampled.to_parquet(f"./{DATASET_NAME}_train_rare_classes_removed_oversampled.parquet", index=False)
print(f"Saved oversampled training set: {len(df_train_oversampled):,} rows")

Saved oversampled training set: 28,756 rows


## 8. Save the size of the data splits as JSON for importing it directly into the report

In [19]:
# show distribution of main labels after oversampling    
splits = {
    "train": ds["train"],
    "validation": ds["validation"],
    "test": ds["test"],
    "train_over": df_train_oversampled,
}

# save dataset stats as JSON for importing it directly into the report
stats = {k: len(v) for k, v in splits.items()}
Path("dataset_stats.json").write_text(json.dumps(stats))

71